# Module 4: SQL-Based Analysis

## Overview

This notebook generates business insights from the cleaned retail transaction dataset using SQL queries.

It produces analytical reports such as Monthly Revenue Analysis, Top Customers, Payment Method Distribution, and Product Category Performance. These reports help organizations understand sales trends and customer behavior.

In [0]:
# Import required PySpark functions

from pyspark.sql.functions import *

# Load cleaned transactions

clean_df = spark.table("trustguard.clean_transactions")

display(clean_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,amount_validation
TXN_8425168,CUST_25,Milk Products,Item_16_milk,27.5,7,192.5,Cash,In-store,2022-03-12,False,Match
TXN_5885894,CUST_01,Patisserie,Item_14_pat,24.5,10,245.0,Credit Card,In-store,2022-02-09,False,Match
TXN_3671271,CUST_18,Patisserie,Item_12_pat,21.5,8,172.0,Cash,In-store,2023-11-27,False,Match
TXN_9908330,CUST_19,Food,Item_18_food,30.5,4,122.0,Credit Card,Online,2022-09-29,True,Match
TXN_2780662,CUST_02,Milk Products,Item_19_milk,32.0,10,320.0,Digital Wallet,In-store,2023-01-19,False,Match
TXN_2646301,CUST_18,Furniture,Item_24_fur,39.5,9,355.5,Credit Card,In-store,2022-12-07,False,Match
TXN_5328604,CUST_07,Food,Item_20_food,33.5,10,335.0,Cash,Online,2024-07-08,False,Match
TXN_8219228,CUST_17,Computers And Electric Accessories,Item_10_cea,18.5,5,92.5,Cash,Online,2022-05-27,False,Match
TXN_1112365,CUST_19,Food,Item_24_food,39.5,10,395.0,Cash,Online,2023-09-24,False,Match
TXN_2953434,CUST_25,Furniture,Item_25_fur,41.0,10,410.0,Credit Card,In-store,2023-08-10,False,Match


In [0]:
# Calculate statistics for quantity

stats = clean_df.selectExpr(
    "avg(quantity) as mean_qty",
    "stddev(quantity) as std_qty"
).collect()[0]

mean_qty = stats["mean_qty"]
std_qty = stats["std_qty"]

print("Mean Quantity :", mean_qty)
print("Standard Deviation :", std_qty)

Mean Quantity : 5.510616302186879
Standard Deviation : 2.790756127805604


In [0]:
# Calculate anomaly threshold

quantity_threshold = mean_qty + (2 * std_qty)

print("Quantity Threshold :", quantity_threshold)

Quantity Threshold : 11.092128557798087


In [0]:
# Detect anomalies

from pyspark.sql.functions import when, lit, col

anomaly_df = (
    clean_df
    .withColumn(
        "anomaly_reason",
        when(
            col("quantity") > quantity_threshold,
            "Quantity exceeds 3 standard deviations"
        )
        .when(
            col("price_per_unit") <= 0,
            "Invalid price"
        )
        .otherwise(None)
    )
    .filter(col("anomaly_reason").isNotNull())
)

print("Total Anomalies :", anomaly_df.count())

display(anomaly_df.limit(10))

Total Anomalies : 0


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,amount_validation,anomaly_reason


In [0]:
# Save anomaly log

anomaly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("trustguard.anomaly_log")

print("Anomaly Log Created Successfully.")

Anomaly Log Created Successfully.


In [0]:
# Verify anomaly log

display(
    spark.table("trustguard.anomaly_log").limit(10)
)

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,amount_validation,anomaly_reason


In [0]:
print("Anomaly DataFrame Count:", anomaly_df.count())
print("Saved Anomaly Table Count:", spark.table("trustguard.anomaly_log").count())

Anomaly DataFrame Count: 0
Saved Anomaly Table Count: 0
